In [1]:
import sys
import numpy as np
from pathlib import Path
import plotly.graph_objects as go

# !mamba install -c conda-forge jupyterlab_widgets ipympl ipywidgets
# %matplotlib widget

In [2]:
PROJECT_ROOT = Path("/Volumes/Data/PycharmProjects/calibration")
sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
import qubic
print(qubic.__file__)
print(qubic.__path__)

/Volumes/Data/PycharmProjects/calibration/qubic/__init__.py
['/Volumes/Data/PycharmProjects/calibration/qubic']


In [4]:
from qubic.lib.Calibration.source_calibration.common.io import read_qubic_dataset, load_tods_from_npz
from qubic.lib.Calibration.source_calibration.common.plotting import plot_az_el_vs_time, plot_focal_plane_tods

In [5]:
def fit_skydip(self, sky_temp, tod):
    pars = np.polyfit(sky_temp, tod, 1)

    return pars[0]

In [ ]:
def skydip_calibration(T_atm: 267,
                       tau_atm:0.1,
                       elevation: np.ndarray,
                       tod: np.ndarray,
                       output_path: Path = None,
                       linewidth: float = 0.8,
                       alpha: float = 0.9):

    z = 0.5 * np.pi - elevation
    airmass = 1 / np.cos(z)
    sky_temp = T_atm * (1 - np.exp(-tau_atm * airmass))

    resp = fit_skydip(sky_temp, tod)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(sky_temp, tod, linewidth=linewidth, alpha=alpha, label=f"{resp:.2f}")
    ax.set_xlabel("Sky Tempo [s]")
    ax.set_ylabel("TOD [ADU]")
    ax.set_title(f"{output_path.parent.parent.name} - TOD TES {tes_idx}")
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()

    if output_path is not None:
        save_path = output_path / f"tes_{tes_idx}.pdf"
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=200)

    plt.show()

    return resp

In [ ]:
def plot_single_tod(tod: np.ndarray,
                    tm: np.ndarray,
                    tes_idx=95,
                    idx=2000,
                    output_path: Path = None,
                    centered: bool = False,
                    normalized: bool = False,
                    linewidth: float = 0.8,
                    alpha: float = 0.9) -> None:
    """
    Plot the TOD of a single TES.
    """
    from matplotlib import pyplot as plt

    name = "raw"

    if centered:
        tod -= np.median(y)
        name = "centered"

    if normalized:
        scale = np.max(np.abs(y))
        if scale == 0:
            scale = 1.0
        tod /= scale

        name = "normalized"

    if not np.any(np.isfinite(y)):
        raise ValueError(f"TES {tes_idx} does not contain any finite samples.")

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(tm, tod, linewidth=linewidth, alpha=alpha)
    ax.axvline(x=tm[idx], color="red", linestyle="--", linewidth=1.5)
    ax.plot(tm[idx], tod[idx], "ro")
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Signal [ADU]")
    ax.set_title(f"{output_path.parent.parent.name} - TOD TES {tes_idx} - {name}")
    ax.grid(True, alpha=0.3)

    fig.tight_layout()

    if output_path is not None:
        save_path = output_path / f"tes_{tes_idx}_{name}.pdf"
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=200)

    plt.show()

In [ ]:
def plot_azimuth_vs_time(tm: np.ndarray,
                         azimuth: np.ndarray,
                         output_path: Path = None,
                         linewidth: float = 0.8,
                         alpha: float = 0.9) -> None:
    """
    Plot azimuth as a function of time.
    """
    from matplotlib import pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(tm, azimuth, linewidth=linewidth, alpha=alpha)
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Azimuth [deg]")
    ax.set_title("Azimuth vs Time")
    ax.grid(True, alpha=0.3)

    fig.tight_layout()

    if output_path is not None:
        output_path.mkdir(parents=True, exist_ok=True)
        save_path = output_path / "azimuth_vs_time.pdf"
        fig.savefig(save_path, dpi=200)

    plt.show()

In [ ]:
def remove_initial_bump(tm: np.ndarray,
                        y: np.ndarray,
                        threshold_frac: float = 0.15,
                        min_stable_samples: int = 30):


    dy = np.abs(np.diff(y, prepend=y[0]))
    thr = threshold_frac * np.nanmax(dy)

    stable = dy < thr

    count = 0
    start_idx = 0
    for i, ok in enumerate(stable):
        if ok:
            count += 1
            if count >= min_stable_samples:
                start_idx = i - min_stable_samples + 1
                break
        else:
            count = 0

    return tm[start_idx:], y[start_idx:], start_idx


In [ ]:
dataset_name = "2026-03-11_16.48.15__SkyDip"
input_dir = Path("/qubic/scripts/Calibration/skydip/data/output") / dataset_name / "dataset"
output_dir = Path("/qubic/scripts/Calibration/skydip/data/output") / dataset_name / "plots"
single_tods_dir = output_dir / "single_tods"

tods_path = input_dir / f"signals_{dataset_name}.npz"
tods = load_tods_from_npz(tods_path)
tm = np.load(input_dir / f"time_{dataset_name}.npy")


azimuth = np.load(input_dir / f"azimuth_{dataset_name}.npy")
elevation = np.load(input_dir / f"elevation_{dataset_name}.npy")

interp_azimuth = np.load(input_dir / f"interp_azimuth_{dataset_name}.npy")
interp_elevation = np.load(input_dir / f"interp_elevation_{dataset_name}.npy")

plot_az_el_vs_time(az=azimuth, el=elevation, output_path=output_dir)

# plot_focal_plane_tods(tods=tods,
#                       output_path=output_dir,
#                       centered=False,
#                       normalized=False,
#                       title=f"Focal-plane TODs - {dataset_name}",
#                       flip_ud=True,
#                       flip_lr=True)

In [ ]:
tes_idx = 95
tod = tods[tes_idx]

In [ ]:
tm, y, start_idx = remove_initial_bump(tm, tod)

In [ ]:
plot_single_tod(tod=y, tm=tm, tes_idx=95, idx=11500, output_path=single_tods_dir, centered=False, normalized=True)

In [ ]:
skydip_calibration(T_atm: 267, tau_atm:0.1, elevation: np.ndarray, tod: np.ndarray, output_path: Path = None, linewidth: float = 0.8, alpha: float = 0.9)

In [ ]:
plot_azimuth_vs_time(tm=tm, azimuth=interp_azimuth, output_path=output_dir)

In [ ]:
tes_index = 95
y = tods[tes_index].astype(float)

skydip_idx_pairs, directions, az_smooth, direction_full = find_skydip_indices_from_azimuth(
    tm=tm,
    azimuth=interp_azimuth,
    smooth_window=101,
    polyorder=3,
    diff_threshold=0.002,
    min_run_length=200,
)

for i, ((start_idx, stop_idx), direction) in enumerate(zip(skydip_idx_pairs, directions), start=1):
    print(
        f"Skydip {i}: direction={direction}, "
        f"start_idx={start_idx}, stop_idx={stop_idx}, "
        f"start_time={tm[start_idx]:.1f}, stop_time={tm[stop_idx]:.1f}"
    )

In [ ]:
plot_azimuth_with_skydip_limits(
    tm=tm,
    azimuth=interp_azimuth,
    az_smooth=az_smooth,
    skydip_idx_pairs=skydip_idx_pairs,
    directions=directions,
    output_path=output_dir,
)

In [ ]:
plot_tod_with_skydip_limits(
    tm=tm,
    y=y,
    skydip_idx_pairs=skydip_idx_pairs,
    directions=directions,
    xx=tes_index,
    normalized=True,
    output_path=output_dir,
)

In [ ]:
# def find_skydip_indices_from_azimuth(tm: np.ndarray,
#                                      azimuth: np.ndarray,
#                                      smooth_window: int = 101,
#                                      polyorder: int = 3,
#                                      diff_threshold: float = 0.002,
#                                      min_run_length: int = 200) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
#     """
#     Find skydip intervals from monotonic parts of the smoothed azimuth.
#
#     A skydip ends when the azimuth stops changing and enters a plateau.
#     """
#     from scipy.signal import savgol_filter
#
#     tm = np.asarray(tm, dtype=float)
#     azimuth = np.asarray(azimuth, dtype=float)
#
#     if tm.ndim != 1 or azimuth.ndim != 1:
#         raise ValueError("tm and azimuth must be 1D arrays.")
#
#     if tm.shape[0] != azimuth.shape[0]:
#         raise ValueError("tm and azimuth must have the same length.")
#
#     if len(azimuth) < 5:
#         raise ValueError("Not enough samples to find skydips.")
#
#     smooth_window = min(smooth_window, len(azimuth))
#     if smooth_window % 2 == 0:
#         smooth_window -= 1
#     if smooth_window <= polyorder:
#         smooth_window = polyorder + 2
#         if smooth_window % 2 == 0:
#             smooth_window += 1
#     if smooth_window >= len(azimuth):
#         smooth_window = len(azimuth) - 1
#         if smooth_window % 2 == 0:
#             smooth_window -= 1
#
#     if smooth_window <= polyorder or smooth_window < 3:
#         raise ValueError("Could not determine a valid smoothing window.")
#
#     az_smooth = savgol_filter(azimuth, window_length=smooth_window, polyorder=polyorder)
#     daz = np.diff(az_smooth, prepend=az_smooth[0])
#
#     direction_full = np.zeros_like(daz, dtype=int)
#     direction_full[daz > diff_threshold] = 1
#     direction_full[daz < -diff_threshold] = -1
#
#     skydip_pairs = []
#     directions = []
#
#     i = 0
#     n = len(direction_full)
#
#     while i < n:
#         # cerca inizio run monotona
#         if direction_full[i] == 0:
#             i += 1
#             continue
#
#         sign = direction_full[i]
#         run_start = i
#
#         # scorri finché resta monotona
#         while i + 1 < n and direction_full[i + 1] == sign:
#             i += 1
#
#         run_stop = i
#
#         # tieni solo run abbastanza lunghe
#         if run_stop - run_start + 1 >= min_run_length:
#             # estendi lo stop fino al primo punto di plateau (direction = 0)
#             stop_idx = run_stop + 1
#             while stop_idx < n and direction_full[stop_idx] == sign:
#                 stop_idx += 1
#
#             skydip_pairs.append([run_start, min(stop_idx, n - 1)])
#             directions.append("up" if sign > 0 else "down")
#
#         i += 1
#
#     if len(skydip_pairs) == 0:
#         return np.empty((0, 2), dtype=int), np.array([], dtype=object), az_smooth, direction_full
#
#     return (
#         np.asarray(skydip_pairs, dtype=int),
#         np.asarray(directions, dtype=object),
#         az_smooth,
#         direction_full,
#     )
#
#
# def plot_azimuth_with_skydip_limits(tm: np.ndarray,
#                                     azimuth: np.ndarray,
#                                     az_smooth: np.ndarray,
#                                     skydip_idx_pairs: np.ndarray,
#                                     directions: np.ndarray,
#                                     output_path: Path = None) -> None:
#     """
#     Plot raw and smoothed azimuth and highlight the skydip start/stop indices.
#     """
#     from matplotlib import pyplot as plt
#
#     tm = np.asarray(tm, dtype=float)
#     azimuth = np.asarray(azimuth, dtype=float)
#     az_smooth = np.asarray(az_smooth, dtype=float)
#     skydip_idx_pairs = np.asarray(skydip_idx_pairs, dtype=int)
#     directions = np.asarray(directions, dtype=object)
#
#     fig, ax = plt.subplots(figsize=(12, 4))
#     ax.plot(tm, azimuth, lw=0.8, alpha=0.4, label="interp azimuth")
#     ax.plot(tm, az_smooth, lw=1.2, label="smoothed azimuth")
#
#     ymin = np.nanmin(az_smooth)
#     ymax = np.nanmax(az_smooth)
#     yrange = ymax - ymin if np.isfinite(ymax - ymin) and (ymax - ymin) > 0 else 1.0
#
#     for i, ((start_idx, stop_idx), direction) in enumerate(zip(skydip_idx_pairs, directions), start=1):
#         ax.axvline(tm[start_idx], color="green", linestyle="--", linewidth=1.5, alpha=0.9)
#         ax.axvline(tm[stop_idx], color="red", linestyle="--", linewidth=1.5, alpha=0.9)
#         mid_time = 0.5 * (tm[start_idx] + tm[stop_idx])
#         ax.text(mid_time, ymax + 0.03 * yrange, f"{i} ({direction})", ha="center", va="bottom", fontsize=8)
#
#     ax.set_xlabel("Time [s]")
#     ax.set_ylabel("Azimuth [deg]")
#     ax.set_title("Azimuth with detected skydip limits")
#     ax.grid(True, alpha=0.3)
#     ax.legend()
#     ax.set_ylim(ymin, ymax + 0.10 * yrange)
#     fig.tight_layout()
#
#     if output_path is not None:
#         output_path.mkdir(parents=True, exist_ok=True)
#         fig.savefig(output_path / "azimuth_with_skydip_limits.pdf", dpi=200)
#
#     plt.show()
#
#
# def plot_tod_with_skydip_limits(tm: np.ndarray,
#                                 y: np.ndarray,
#                                 skydip_idx_pairs: np.ndarray,
#                                 directions: np.ndarray | None = None,
#                                 tes_indices: int | None = None,
#                                 normalized: bool = False,
#                                 centered: bool = False,
#                                 output_path: Path = None) -> None:
#     """
#     Plot a TOD and overlay start/stop indices of skydips found from azimuth.
#     """
#     from matplotlib import pyplot as plt
#
#     tm = np.asarray(tm, dtype=float)
#     y_plot = np.asarray(y, dtype=float).copy()
#     skydip_idx_pairs = np.asarray(skydip_idx_pairs, dtype=int)
#
#     if tm.ndim != 1 or y_plot.ndim != 1:
#         raise ValueError("tm and y must be 1D arrays.")
#
#     if tm.shape[0] != y_plot.shape[0]:
#         raise ValueError("tm and y must have the same length.")
#
#     if skydip_idx_pairs.ndim != 2 or skydip_idx_pairs.shape[1] != 2:
#         raise ValueError("skydip_idx_pairs must have shape (n_skydips, 2).")
#
#     if centered:
#         y_plot -= np.nanmedian(y_plot)
#
#     if normalized:
#         scale = np.nanmax(np.abs(y_plot))
#         if not np.isfinite(scale) or scale == 0:
#             scale = 1.0
#         y_plot /= scale
#
#     fig, ax = plt.subplots(figsize=(12, 4))
#     ax.plot(tm, y_plot, lw=1.0)
#
#     ymin = np.nanmin(y_plot)
#     ymax = np.nanmax(y_plot)
#     yrange = ymax - ymin if np.isfinite(ymax - ymin) and (ymax - ymin) > 0 else 1.0
#
#     if directions is None:
#         directions = np.array([""] * len(skydip_idx_pairs), dtype=object)
#     else:
#         directions = np.asarray(directions, dtype=object)
#
#     for i, ((start_idx, stop_idx), direction) in enumerate(zip(skydip_idx_pairs, directions), start=1):
#         ax.axvline(tm[start_idx], color="green", linestyle="--", linewidth=1.5, alpha=0.9)
#         ax.axvline(tm[stop_idx], color="red", linestyle="--", linewidth=1.5, alpha=0.9)
#         mid_time = 0.5 * (tm[start_idx] + tm[stop_idx])
#         label = f"{i}" if direction == "" else f"{i} ({direction})"
#         ax.text(mid_time, ymax + 0.03 * yrange, label, ha="center", va="bottom", fontsize=8)
#
#     title = "TOD with skydip limits from azimuth"
#     if tes_indices is not None:
#         title += f" - TES {tes_indices}"
#
#     ax.set_title(title)
#     ax.set_xlabel("Time [s]")
#     ax.set_ylabel("Signal [ADU]")
#     ax.grid(True, alpha=0.3)
#     ax.set_ylim(ymin, ymax + 0.08 * yrange)
#     fig.tight_layout()
#
#     if output_path is not None:
#         output_path.mkdir(parents=True, exist_ok=True)
#         fig.savefig(output_path / "tod_with_skydip_limits_from_azimuth.pdf", dpi=200)
#
#     plt.show()
#
#
# def plot_single_skydip_segment_from_indices(tm: np.ndarray,
#                                             y: np.ndarray,
#                                             start_idx: int,
#                                             stop_idx: int,
#                                             skydip_id: int | None = None,
#                                             direction: str | None = None,
#                                             tes_indices: int | None = None,
#                                             normalized: bool = False,
#                                             centered: bool = False,
#                                             output_path: Path = None) -> None:
#     """
#     Plot a single skydip segment from start/stop indices.
#     """
#     from matplotlib import pyplot as plt
#
#     tm = np.asarray(tm, dtype=float)
#     y_plot = np.asarray(y, dtype=float).copy()
#
#     if start_idx < 0 or stop_idx >= len(tm) or stop_idx <= start_idx:
#         raise ValueError("Invalid start_idx/stop_idx for skydip segment.")
#
#     seg_tm = tm[start_idx:stop_idx + 1]
#     seg_y = y_plot[start_idx:stop_idx + 1]
#
#     if centered:
#         seg_y -= np.nanmedian(seg_y)
#
#     if normalized:
#         scale = np.nanmax(np.abs(seg_y))
#         if not np.isfinite(scale) or scale == 0:
#             scale = 1.0
#         seg_y /= scale
#
#     fig, ax = plt.subplots(figsize=(8, 4))
#     ax.plot(seg_tm, seg_y, lw=1.0)
#     ax.set_xlabel("Time [s]")
#     ax.set_ylabel("Signal [ADU]")
#
#     title = f"skydip: {start_idx} -> {stop_idx}"
#     if skydip_id is not None:
#         title = f"skydip {skydip_id} - " + title
#     if direction is not None:
#         title += f" - {direction}"
#     if tes_indices is not None:
#         title = f"TES {tes_indices} - " + title
#
#     ax.set_title(title)
#     ax.grid(True, alpha=0.3)
#     fig.tight_layout()
#
#     if output_path is not None:
#         output_path.mkdir(parents=True, exist_ok=True)
#         suffix = f"_{skydip_id:02d}" if skydip_id is not None else ""
#         fig.savefig(output_path / f"skydip_from_azimuth{suffix}.pdf", dpi=200)
#
#     plt.show()